In [ ]:
from typing import Iterable, Self, Any
from collections.abc import Generator
from itertools import chain as iterchain, combinations as itercomb
from collections import deque

iterchainiter = iterchain.from_iterable

In [ ]:
from board import DIGITS, POS9, Cell, Node, Board, Loc, Locality, Target, MultiTarget
from utils import iter_allzones, countfinals, validate
from solving import orchestrator, solver, Resolution, Resolving, Resolver

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
async def solve(initial: Board, *resolvers):
    return await solver(initial, orchestrator(initial, *resolvers))

## Basic

Singles in localities and their contraneighbours


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for finode in filter(lambda n: n.cell.is_final, board):
        assert finode.cell.final is not None
        findig = finode.cell.final
        targ = Target(finode.loc, findig)
        around = Locality.around(targ.loc)
        allaround: Iterable[Loc] = list(iterchain.from_iterable(zone.locs() for zone in around))
        spoilers: Iterable[Node] = set(board.drafts(allaround, lambda n: findig in n.cell))
        castaways = set(Target(n.loc, targ.dig) for n in spoilers)
        if castaways:
            yield Resolution(
                castaways,
                set(),
                highlights={"anchors": {targ}},
            )

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    for dig in DIGITS:
        for zone in iter_allzones():
            neighborhood = list(board.drafts(zone))
            family = list(filter(lambda n: dig in n.cell, neighborhood))
            if len(family) == 1:
                lonesome = family[0]
                if len(lonesome.cell) > 1:
                    yield Resolution(
                        set(),
                        {Target(lonesome.loc, dig)},
                        highlights={"anchors": set(Target(n.loc, dig) for n in neighborhood if n != lonesome)},
                    )

## Multiples

Combos of N digits

### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

=> remove the digits from all other cell of the locality

### hidden

Some n-combo contained in only n cells (within locality) // along other drafts

=> remove other drafts from the cells => it becomes open


In [ ]:
def gen_combos(m: int):
    return map(set[int], itercomb(DIGITS, m))

In [ ]:
def openmults(board: Board, mult: int) -> Resolving:
    for zone in iter_allzones():
        neighborhood = tuple(board.drafts(zone))
        for combo in gen_combos(mult):
            habitat = tuple(filter(lambda n: n.cell <= combo, neighborhood))
            neighbors = tuple(filter(lambda n: n not in habitat and n.cell & combo, neighborhood))
            if len(habitat) == mult and len(neighbors) > mult:
                castaways = set(Target(n.loc, d) for n in neighbors for d in n.cell & combo)
                anchors = set(MultiTarget(n.loc, n.cell & combo) for n in habitat)
                yield Resolution(
                    castaways,
                    set(),
                    highlights={"anchors": anchors},
                )


def openmults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return openmults(board, mult)

    resolver.__name__ = f"openmults[{mult}]"
    return resolver

In [ ]:
def unhidemults(board: Board, mult: int) -> Resolving:
    for zone in iter_allzones():
        neighborhood = tuple(board.drafts(zone))
        for combo in gen_combos(mult):
            habitat = tuple(filter(lambda n: len(n.cell & combo), neighborhood))
            habitants = set(iterchainiter(n.cell & combo for n in habitat))
            spoiled = tuple(filter(lambda n: len(n.cell - combo), habitat))
            if len(habitat) == mult and len(habitants) == mult and len(spoiled) > 0:
                castaways = set(Target(n.loc, d) for n in spoiled for d in n.cell - combo)
                anchors = set(MultiTarget(n.loc, n.cell & combo) for n in habitat)
                yield Resolution(
                    castaways,
                    set(),
                    highlights={"anchors": anchors},
                )


def unhidemults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return unhidemults(board, mult)

    resolver.__name__ = f"unhidemults[{mult}]"
    return resolver

## Links

### hard links

Represent XOR relation

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell

The criteria are totally independent of board content (assuming target digits exist)

### chains

Alterating link chains: (-xor-nand-)^n

(A-xor-B-nand-)^n-xor-D and (A-nand-D) => (A-xor-D)

All {x: (x-nand-A) and (x-nand-D)} can be eliminated


In [ ]:
class Link(tuple[Target, Target]):
    """Ordered immutable pair of targets"""

    # ordered for chains

    def __str__(self):
        return f"{self[0]} ~ {self[1]}"

    def strtail(self):
        return f" ~ {self[1]}"

    def __repr__(self):
        return f"{self.__class__.__name__}(({self[0]!r}, {self[1]!r},))"

    def reversed(self):
        return self.__class__((self[1], self[0]))

    def __hash__(self):
        # unordered hash
        return hash(frozenset((self)))

    def __eq__(self, other):
        return hash(self) == hash(other)


class HLink(Link):
    """Hard link, XOR relation"""

    def __str__(self):
        return f"{self[0]}⟺{self[1]}"

    def strtail(self):
        return f"⟺{self[1]}"


class SLink(Link):
    """Soft link, NAND relation"""

    def __str__(self):
        return f"{self[0]}⟷{self[1]}"

    def strtail(self):
        return f"⟷{self[1]}"

In [ ]:
def scan_hard(board: Board) -> Generator[tuple[Target, Target]]:

    def scan_cell(loc: Loc):
        node = board.get(loc)
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield (
                Target(node.loc, d1),
                Target(node.loc, d2),
            )

    def scan_locality(zone: Locality):
        for d in DIGITS:
            family = tuple(board.drafts(zone, lambda n: d in n.cell))
            if len(family) == 2:
                n1, n2 = family
                yield (
                    Target(n1.loc, d),
                    Target(n2.loc, d),
                )

    for node in board.drafts():
        yield from scan_cell(node.loc)

    for i in POS9:
        yield from scan_locality(Locality(i, ..., ...))
        yield from scan_locality(Locality(..., i, ...))
        yield from scan_locality(Locality(..., ..., i))


In [ ]:
def check_soft(t1: Target, t2: Target):
    l1 = t1.loc
    l2 = t2.loc
    if t1.dig == t2.dig:
        return l1.blk == l2.blk or l1.row == l2.row or l1.col == l2.col
    else:
        return l1 == l2


def check_linksoft(lnk1: Link, t2: Target):
    return check_soft(lnk1[0], t2) and check_soft(lnk1[1], t2)

In [ ]:
def search_links(board: Board) -> set[HLink]:
    return set(set[HLink](HLink(targets) for targets in scan_hard(board)))

In [ ]:
import re


class Chain(tuple[Link, ...]):
    @classmethod
    def init(cls, link: Link):
        return cls((link,))

    def __str__(self):
        return "".join([str(self[0])] + [lnk.strtail() for lnk in self[1:]])

    def __add__(self, other: Self):
        assert self[-1][-1] == other[0][0]
        return Chain(tuple(self) + tuple(other))

    def anchors(self) -> Iterable[Target]:
        """All anchor points in the chain"""
        return (self[0][0], *(lnk[1] for lnk in self))

    def ends(self):
        return (self[0][0], self[-1][1])

    def __hash__(self):
        """Hashing by unordered links"""
        return hash(frozenset(self))

    def __eq__(self, other: Self):
        return hash(self) == hash(other)

    @property
    def is_cyclic(self):
        e1, e2 = self.ends()
        return e1 == e2

    def pattern(self):
        """Returns string pattern of link classes like `HLink~SLink~`"""
        kinds = [lnk.__class__.__name__ for lnk in self]
        pattern = "~".join(kinds)
        if self.is_cyclic:
            return f"~{pattern}~"
        else:
            return pattern

In [ ]:
RE_ALC = re.compile(r"^(HLink~SLink~)+HLink$")
RE_ALCl = re.compile(r"^~(HLink~SLink~)+$")


def check_goal(chain: Chain, links: Iterable[Link]):
    """Check if the chain is suited for resolvation"""
    # chould be cyclic and contain some non-xor links
    return RE_ALCl.match(chain.pattern()) and any(lnk not in links for lnk in chain)


def check_expansion(last: HLink, other: HLink) -> tuple[SLink, HLink] | None:
    front = last[1]
    if front in other:
        return None
    if check_soft(front, other[0]):
        return SLink((front, other[0])), other
    if check_soft(front, other[1]):
        return SLink((front, other[1])), other.reversed()


def expand_alc(chain: Chain, links: Iterable[HLink]) -> Generator[Chain]:
    last: HLink = chain[-1]  # type: ignore

    e1, e2 = chain.ends()

    if e1 != e2 and check_soft(e1, e2):
        yield chain + Chain((SLink((e2, e1)),))

    for other in links:
        if other not in chain:
            expansion = check_expansion(last, other)
            if expansion is not None and expansion[0] not in chain:
                yield chain + Chain(expansion)


def find_chain_d(links: Iterable[HLink]) -> Chain | None:
    """Find a longest chain"""
    # depth-first graph search (longest chain first)
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.pop()
        if check_goal(chain, links):
            return chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def find_chain_s(links: Iterable[HLink]) -> Chain | None:
    """Find a shortest chain"""
    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if check_goal(chain, links):
            return chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)
    return None


def search_chains(links: Iterable[HLink]) -> Generator[Chain]:
    """Search for all chains"""
    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if check_goal(chain, links):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)
    return None

In [ ]:
def resolve_chains1(current: Board) -> Resolving:
    links = search_links(current)

    # resolving first found chain
    chain = find_chain_s(links)
    if chain is None:
        return

    for link in chain:
        t1, t2 = link
        for zone in Locality.common(t1.loc, t2.loc):
            # print(link, zone, "...")
            neighbors = current.drafts(zone)
            castaways = set(trg for node in neighbors for trg in node if trg not in link and check_linksoft(link, trg))
            if len(castaways):
                yield Resolution(
                    castaways,
                    set(),
                    highlights={"anchors": {t1, t2}, "links": set(chain)},
                )

## A puzzle


In [ ]:
from utils import parse

# simple: solvable by basics
puzzle = parse("""
5..74...2
17..8..59
283.1.467
6.84..173
9....82..
7.2.3..86
8.....79.
39.86152.
..59.....
""")

# expert level: solvable by basics + multiples
# puzzle = parse("""
# ....8.41.
# 6........
# .29..58..
# 8...7.2..
# .........
# .7......5
# 2...3...8
# ...5...3.
# .4.7.9..6
# """)

# extreme level: not solvable by myself
# puzzle = parse("""
# .8.....52
# .......87
# ....98...
# 4...3.6..
# .2.7.....
# .........
# 6..8.2...
# ...5.91..
# 9........
# """)


puzzle = Board.transform(puzzle, fillempty)

### UI

async ui with inspection of resolutions


In [ ]:
# widgets

from collections import Counter
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas


def format_options(objects: Iterable[Any]):
    return tuple((str(obj), obj) for obj in objects)


def on_change(widget: w.Widget):
    def wrapper(handler):
        return widget.observe(handler, "value")

    return wrapper


debug_view = w.Output()

LINKSTYLES = {"HLink": "HARD", "SLink": "SOFT"}
column_layout = w.Layout(width="auto", height="100%", flex_flow="column", align_items="stretch")


canvas = SudokuCanvas()


def deselect(widget):
    widget.value = () if isinstance(widget, w.SelectMultiple) else None


@canvas.on_client_ready
def init_canvas():
    canvas[2].global_alpha = 0.5
    canvas.draw_grid()


@canvas.on_mouse_up
def on_canvas_click(x, y):
    # targ = canvas.map_target(x, y)
    pass


def highlight_targets(targets: list[Target] | list[MultiTarget], color: str):
    for trg in targets:
        if isinstance(trg, Target):
            canvas.highlight_target(trg, color=color)
        if isinstance(trg, MultiTarget):
            canvas.highlight_group(trg, color=color)


def highlight_links(links: list[Link], color: str):
    for lnk in links:
        canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__], color=color)


def wCounterText(label: str):
    return w.Text(
        label,
        layout=dict(width="auto"),
    )


counters_labels = {dig: wCounterText("...") for dig in DIGITS}
countotal_label = wCounterText("...")


def show_counters(counters: Counter):
    for dig, cnt in counters.items():
        counters_labels[dig].value = f"({dig}): {cnt}"
        counters_labels[dig].style.background = "var(--jp-success-color2)" if cnt == 9 else ""
    total = counters.total()
    countotal_label.value = f"Total: {total}"
    countotal_label.style.background = "var(--jp-success-color1)" if total == 81 else ""


status_label = w.Text(layout=dict(width="auto"), style=dict(text_color="white"))


def show_status(status: str):
    status_label.value = status
    status_label.style.visibility = "visible"
    if status == "SOLVED":
        status_label.style.background = "var(--jp-success-color0)"
    elif status == "BROKEN":
        status_label.style.background = "var(--jp-error-color0)"
    else:
        status_label.style.background = "var(--jp-info-color3)"


resolver_label = w.Label(value="")
resolver_count = w.Label(value="")

btn_running = w.Button(icon="gear spin", button_style="warning", style=dict(font_size="large"), layout=dict(visibility="hidden"))
btn_continue = w.Button(description="Continue", disabled=True, button_style="primary")
select_stepforw = w.SelectMultiple(
    options=[],
    description="Inspecting:",
    value=[],
    layout=dict(flex_flow="column", width="auto", align_items="flex-start"),
    indent=False,
    style=dict(description_width="auto", text_align="left"),
    rows=20,
)


column_layout = w.Layout(
    width="auto",
    height="auto",
    flex_flow="column",
    justify_content="flex-start",
    align_items="stretch",
    margin="0 4px",
    overflow="hidden",
)

selecting_anchors = w.SelectMultiple(
    description="Anchors",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)

selecting_links = w.SelectMultiple(
    description="Links",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)

selecting_chains = w.SelectMultiple(
    description="Chains",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)


@on_change(selecting_anchors)
def on_selecting_anchors(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Target] = list(change.new)
        with hold_canvas():
            highlight_targets(selected, "blue")


@on_change(selecting_links)
@debug_view.capture()
def on_selecting_links(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Link] = list(change.new)
        # print("links selected", list(map(str, selected)))
        with hold_canvas():
            highlight_links(selected, "blue")
            anchors = set(iterchainiter(selected))
            selecting_anchors.value = list(anchors)


@on_change(selecting_chains)
@debug_view.capture()
def on_selecting_chains(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Chain] = list(change.new)
        # print("chains selected", list(map(str, selected)))
        with hold_canvas():
            links = set(iterchainiter(selected))
            hlinks = list(filter(lambda l: isinstance(l, HLink), links))
            slinks = list(filter(lambda l: isinstance(l, SLink), links))
            selecting_links.value = hlinks
            highlight_links(slinks, color="blue")

In [ ]:
# public methods kinda


def draw_board(board: Board):
    canvas.draw_board(board)


def show_targets(targets: Iterable[Target]):
    highlight_targets(list(targets), "orange")


def show_finals(targets: Iterable[Target]):
    with hold_canvas():
        for trg in targets:
            canvas.highlight_final(trg, color="purple")


def show_anchors(anchors: Iterable[Target] | set[MultiTarget], select: bool = False):
    selecting_anchors.options = format_options(anchors)
    if select:
        selecting_anchors.value = list(anchors)


def show_links(links: Iterable[Link], select: bool = False):
    selecting_links.options = format_options(links)
    if select:
        selecting_anchors.value = []
        selecting_links.value = list(links)


def show_chains(chains: Iterable[Chain], select: bool = False):
    selecting_chains.options = format_options(chains)
    if select:
        selecting_anchors.value = []
        selecting_links.value = []
        selecting_chains.value = list(chains)


btn_reload = w.Button(description="Reload")


@btn_reload.on_click
def on_reload(btn):
    canvas.clear_highlights()
    draw_board(puzzle)
    show_counters(countfinals(puzzle))
    show_status(validate(puzzle))

In [ ]:
display(
    w.HBox(
        [
            w.VBox(
                [w.Label("Status"), *counters_labels.values(), countotal_label, status_label],
                layout=dict(align_items="stretch", width="6em"),
            ),
            canvas,
            w.VBox([
                btn_reload,
                w.HBox([resolver_label, resolver_count]),
                btn_running,
                btn_continue,
                select_stepforw,
            ]),
            selecting_anchors,
            selecting_links,
            selecting_chains,
        ],
        layout=dict(justify_content="flex-start", align_items="stretch"),
    )
)

In [ ]:
debug_view

In [ ]:
import asyncio
from typing import AsyncGenerator


def wait_continue():
    btn_continue.disabled = False
    future = asyncio.Future()

    def on_click(b):
        btn_continue.on_click(on_click, remove=True)
        btn_continue.disabled = True
        future.set_result(True)

    btn_continue.on_click(on_click)
    return future


async def solver_ui(initial: Board, orchestra: AsyncGenerator[Resolver, Board]):
    """Integrated with UI"""
    resolver = await orchestra.asend(None)  # type: ignore that fucking caveat

    current = initial
    render_result(current)
    while True:
        resolver_label.value = resolver.__name__
        for resolution in resolver(current):
            btn_running.layout.visibility = "hidden"
            resolver_count.value = f"-{len(resolution.castaways)} +{len(resolution.finals)}"
            stepping = resolver.__name__ in select_stepforw.value

            if stepping:
                render_resolution(resolution)
                await wait_continue()
                clear_resolution()

            current = resolution.apply(current)

            render_result(current)
            btn_running.layout.visibility = "visible"
            resolver_count.value = "..."
            if stepping:
                await asyncio.sleep(0.2)
        try:
            resolver = await orchestra.asend(current)
        except StopAsyncIteration:
            break
    resolver_label.value = ""
    resolver_count.value = ""
    btn_running.layout.visibility = "hidden"
    return current


async def solve_ui(*resolvers):
    global puzzle
    select_stepforw.options = [r.__name__ for r in resolvers]
    select_stepforw.value = select_stepforw.options[:]
    puzzle = await solver_ui(puzzle, orchestrator(puzzle, *resolvers))
    select_stepforw.options = []
    show_status(validate(puzzle))


def render_resolution(res: Resolution):
    with hold_canvas():
        show_targets(res.castaways)
        show_finals(res.finals)
        if res.highlights is not None and "anchors" in res.highlights:
            show_anchors(res.highlights["anchors"], select=True)
        if res.highlights is not None and "links" in res.highlights:
            show_links(res.highlights["links"], select=True)


def clear_resolution():
    canvas.clear_highlights()


def render_result(result: Board):
    with hold_canvas():
        draw_board(result)
        show_counters(countfinals(result))


def run(task):
    return asyncio.create_task(task)

In [ ]:
# links: list[HLink] = list(search_links(puzzle))
# chains = list(search_chains(links))
# anchors = set(iterchainiter(links))
# show_anchors(anchors)
# show_links(links)
# show_chains(chains)

In [ ]:
task = run(
    solve_ui(
        singles,
        cleanup,
        # openmults_(2),
        # unhidemults_(2),
        # openmults_(3),
        # unhidemults_(3),
        # openmults_(4),
        # unhidemults_(4),
        # resolve_chains1,
    )
)

In [ ]:
task.cancel()

In [ ]:
status_label.style.background = "red"

In [ ]:
validate(puzzle)